# Module 3: Vision Transformers in Keras

> **Assignment (10 points):** Extract CNN feature maps, feed them into a Vision Transformer, build and train a CNN-ViT hybrid model.

In [ ]:
import os, sys, pathlib, warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
PROJECT_ROOT = pathlib.Path(os.path.abspath(os.getcwd()))
for p in [PROJECT_ROOT] + list(PROJECT_ROOT.parents):
    if (p / 'images_dataSAT').is_dir():
        PROJECT_ROOT = p
        break
os.chdir(PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

tf.random.set_seed(42)

### Task 1: Load the pre-trained CNN model in `cnn_model` using `load_model()` and print the summary

In [ ]:
cnn_model = tf.keras.models.load_model('models/cnn_model.keras')
cnn_model.summary()

### Task 2: Based on `model.summary()`, get the name of the layer for feature extraction in `feature_layer_name`

In [ ]:
feature_layer_name = [l.name for l in cnn_model.layers
                      if isinstance(l, tf.keras.layers.MaxPooling2D)][-1]
print('Feature extraction layer:', feature_layer_name)

# Rebuild the CNN as a feature extractor that ends at feature_layer_name,
# reusing the exact same (pretrained) layer instances.
inputs = tf.keras.Input(shape=(64, 64, 3))
x = inputs
for layer in cnn_model.layers:
    x = layer(x)
    if layer.name == feature_layer_name:
        break
feature_extractor = tf.keras.Model(inputs=inputs, outputs=x)
feature_extractor.trainable = False
feature_extractor.summary()

## CNN-ViT hybrid architecture (Task 3)

In [ ]:
def mlp_block(x, hidden_units, dropout_rate):
    for units in hidden_units:
        x = tf.keras.layers.Dense(units, activation='gelu')(x)
        x = tf.keras.layers.Dropout(dropout_rate)(x)
    return x


def transformer_encoder_block(x, embed_dim, num_heads, ff_dim, dropout_rate=0.1):
    residual = x
    x = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)
    x = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim, dropout=dropout_rate)(x, x)
    x = tf.keras.layers.Dropout(dropout_rate)(x)
    x = tf.keras.layers.Add()([residual, x])

    residual = x
    x = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)
    x = mlp_block(x, [ff_dim, embed_dim], dropout_rate)
    x = tf.keras.layers.Add()([residual, x])
    return x


def build_cnn_vit_hybrid(feature_extractor, embed_dim=128, num_heads=4, ff_dim=256,
                         num_transformer_layers=4, dropout_rate=0.1, num_classes=1):
    inputs = tf.keras.Input(shape=(64, 64, 3))
    feature_maps = feature_extractor(inputs)          # (None, 4, 4, 256)

    h, w, c = feature_maps.shape[1], feature_maps.shape[2], feature_maps.shape[3]
    num_patches = h * w                                # 16 tokens
    patches = tf.keras.layers.Reshape((num_patches, c))(feature_maps)

    x = tf.keras.layers.Dense(embed_dim)(patches)      # patch embedding
    positions = tf.range(start=0, limit=num_patches, delta=1)
    pos_embedding = tf.keras.layers.Embedding(input_dim=num_patches, output_dim=embed_dim)(positions)
    x = x + pos_embedding

    for _ in range(num_transformer_layers):
        x = transformer_encoder_block(x, embed_dim, num_heads, ff_dim, dropout_rate)

    x = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = tf.keras.layers.Dropout(dropout_rate)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='sigmoid')(x)

    return tf.keras.Model(inputs=inputs, outputs=outputs, name='cnn_vit_hybrid')


hybrid_model = build_cnn_vit_hybrid(feature_extractor)
hybrid_model.summary()

### Task 4: Compile `hybrid_model`

In [ ]:
hybrid_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

### Task 5: Define the training configuration of `hybrid_model`

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rescale=1. / 255, rotation_range=20, width_shift_range=0.2,
    height_shift_range=0.2, zoom_range=0.2, horizontal_flip=True,
    validation_split=0.2,
)
train_generator = datagen.flow_from_directory(
    './images_dataSAT', target_size=(64, 64), batch_size=32, class_mode='binary',
    subset='training', shuffle=True, seed=42)
validation_generator = datagen.flow_from_directory(
    './images_dataSAT', target_size=(64, 64), batch_size=32, class_mode='binary',
    subset='validation', shuffle=False, seed=42)

EPOCHS = 5
BATCH_SIZE = 32

os.makedirs('models', exist_ok=True)
checkpoint_vit = tf.keras.callbacks.ModelCheckpoint(
    'models/vit_hybrid.keras', monitor='val_accuracy', mode='max', save_best_only=True)
early_stop_vit = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=3, restore_best_weights=True)

history_vit = hybrid_model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=EPOCHS,
    callbacks=[checkpoint_vit, early_stop_vit],
    verbose=1,
)

## Plot losses and save the hybrid model

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history_vit.history['loss'], label='Training Loss')
plt.plot(history_vit.history['val_loss'], label='Validation Loss')
plt.title('Hybrid CNN-ViT Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.savefig('models/vit_hybrid_loss.png')
plt.show()

hybrid_model.save('models/vit_hybrid.keras')
print('Hybrid model saved -> models/vit_hybrid.keras')